# 33. CI for AI

**Tier:** Production & Safety
**Estimated time:** 45 minutes
**Prerequisites:** 24, 27, 28
**Priority:** 🟡 Important — eval gates in CI turn notebook-24 skills into team-level leverage; important, but it's plumbing around the crucial skill (evals) rather than the skill itself. *If skipped, revisit when:* a second person starts editing your prompts, or after the first "the prompt change broke prod" incident.
**Source material:** This repo's own `.claude/checks/` + `/loop` skeleton — a working example of exactly this pattern

## What You'll Learn
- Versioning prompts like code, so a prompt change is a reviewable diff, not an invisible edit
- Wiring the notebook-24 eval harness into a regression gate that runs automatically
- A GitHub Actions workflow that blocks a merge on an eval regression
- Canary rollout policy: gating a full rollout on a measured score, not a vibe

## Why This Matters
Every eval harness and agent-eval suite built in this curriculum is only as useful as the discipline that runs it automatically, on every change, before it reaches production. Without CI enforcement, "we have an eval harness" quietly becomes "we ran it once, six months ago." This notebook takes the harness from notebook 24 and wires it into the same gate that makes a codebase's test suite trustworthy — and it uses this very repo's own `.claude/checks/` + `/loop` skeleton as the worked example, since that's exactly this pattern already running.


In [ ]:
import os, json, subprocess, pathlib, tempfile
import numpy as np

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — live eval-gate cells will be skipped.")

def ask(prompt, system="Answer directly.", max_tokens=60, temperature=0.0):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]"
    msg = client.messages.create(model=TEACH_MODEL, max_tokens=max_tokens, system=system,
                                  temperature=temperature, messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text


## Version prompts like code

A prompt (or system message, or few-shot set) is a production artifact with the same failure modes as code: it can regress, it can be edited without review, and it can drift silently across environments. Treat it exactly like code — in a file, in version control, reviewed via a diff — instead of as a string that lives only inside a notebook cell or a database row somewhere.

In [ ]:
PROMPTS_DIR = pathlib.Path(tempfile.mkdtemp(prefix="nb33_prompts_"))

# "main" branch prompt — the currently-shipped version.
(PROMPTS_DIR / "system_prompt_v1.txt").write_text(
    "Answer the question directly. If arithmetic is involved, double-check your answer."
)
# "PR branch" prompt — a proposed change under review.
(PROMPTS_DIR / "system_prompt_v2.txt").write_text(
    "Answer the question directly and concisely. If arithmetic is involved, show your work briefly."
)

print("Prompt files (this is what a PR diff would show):")
print(" v1:", (PROMPTS_DIR / "system_prompt_v1.txt").read_text())
print(" v2:", (PROMPTS_DIR / "system_prompt_v2.txt").read_text())


## Wiring the notebook-24 harness into a regression gate

Reuse the exact eval-harness shape from notebook 24 — golden dataset + scorer — but now as a gate function that returns pass/fail, not just a printed score. This is the function a CI job calls; everything before it in this notebook is just building the inputs to this one decision.

In [ ]:
import re

def normalize(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def exact_match_score(output, reference):
    return 1.0 if normalize(reference) in normalize(output) else 0.0

FROZEN_GOLDEN = [
    {"q": "What is 12 + 7?", "ref": "19"},
    {"q": "What is 9 * 6?", "ref": "54"},
    {"q": "What is the capital of Spain?", "ref": "Madrid"},
    {"q": "What is 100 - 37?", "ref": "63"},
]

def eval_prompt_file(prompt_path):
    system_prompt = pathlib.Path(prompt_path).read_text()
    scores = [exact_match_score(ask(item["q"], system=system_prompt, max_tokens=60), item["ref"])
              for item in FROZEN_GOLDEN]
    return np.mean(scores)

def regression_gate(current_prompt_path, candidate_prompt_path, tolerance=0.0):
    """Returns (passed: bool, current_score, candidate_score). This is what CI calls."""
    current_score = eval_prompt_file(current_prompt_path)
    candidate_score = eval_prompt_file(candidate_prompt_path)
    passed = candidate_score >= current_score - tolerance
    return passed, current_score, candidate_score

passed, current_score, candidate_score = regression_gate(
    PROMPTS_DIR / "system_prompt_v1.txt", PROMPTS_DIR / "system_prompt_v2.txt"
)
print(f"current (v1) score:   {current_score:.0%}")
print(f"candidate (v2) score: {candidate_score:.0%}")
print(f"GATE {'PASSED — safe to merge' if passed else 'FAILED — blocks merge'}")


## A GitHub Actions workflow that enforces the gate

This is the same job shape as `.claude/checks/notebooks.sh` in this repo — a script that prints PASS/FAIL and exits non-zero on failure — wired into a CI trigger instead of run manually. The workflow below would live at `.github/workflows/eval-gate.yml` in a real repo; a pull request that regresses the frozen golden dataset score fails the check and cannot merge.

In [ ]:
GITHUB_ACTIONS_YAML = '''
name: Eval Regression Gate

on:
  pull_request:
    paths:
      - "prompts/**"

jobs:
  eval-gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.13"
      - run: pip install -r requirements.txt
      - name: Run eval regression gate
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: |
          python ci/eval_gate.py \
            --current prompts/system_prompt.txt \
            --candidate ${{ github.event.pull_request.head.sha }}:prompts/system_prompt.txt
          # eval_gate.py exits non-zero on regression, which fails this CI job
          # and blocks the merge via a required-status-check branch protection rule.
'''
print(GITHUB_ACTIONS_YAML)


## The pattern this repo already runs

This curriculum's own `.claude/checks/` directory is a smaller-scale, notebook-specific version of exactly this idea: `notebooks.sh` is a goal check that prints PASS/FAIL and exits non-zero on failure, and `/loop` (`.claude/commands/loop.md`) repeatedly does-work → runs-the-check → repeats until it's green — the same do/check/repeat discipline a CI pipeline enforces on every PR, just running locally instead of on a server. The mapping is direct:

| This notebook | This repo |
|---|---|
| `regression_gate()` | `.claude/checks/notebooks.sh` |
| frozen `FROZEN_GOLDEN` dataset | the notebooks themselves, executed clean |
| GitHub Actions job | `/loop` calling the check script |
| "blocks merge on regression" | "loop repeats until the check passes, stops after 5 cycles" |

If you understand why this repo never lets a notebook ship without `notebooks.sh` passing, you already understand why a production LLM feature should never ship without its eval gate passing.

## Canary rollout policy

A regression gate answers "is this safe to merge?" A canary rollout answers the next question: "is this safe to give 100% of traffic to, right now?" Even a change that passes the gate should typically roll out gradually — 5% of traffic, then 25%, then 100% — with the SAME eval harness (now scored against live traffic samples, not just the frozen dataset) monitored at each stage, exactly as notebook 27 covered for production monitoring.

In [ ]:
ROLLOUT_STAGES = [0.05, 0.25, 1.00]

def canary_rollout(candidate_prompt_path, live_traffic_sample, min_score=0.8):
    """Simulate staged rollout: advance a stage only if the candidate holds up on live-shaped
    traffic, not just the frozen golden set (which the prompt may have been tuned against)."""
    for stage_pct in ROLLOUT_STAGES:
        n_sampled = max(1, int(len(live_traffic_sample) * stage_pct))
        sample = live_traffic_sample[:n_sampled]
        system_prompt = pathlib.Path(candidate_prompt_path).read_text()
        scores = [exact_match_score(ask(item["q"], system=system_prompt), item["ref"]) for item in sample]
        stage_score = np.mean(scores) if scores else 0.0
        print(f"stage {stage_pct:.0%} traffic (n={n_sampled}): score={stage_score:.0%}")
        if stage_score < min_score:
            print(f"  -> HALT rollout at {stage_pct:.0%}: score below {min_score:.0%} threshold.")
            return stage_pct
    print("  -> Rollout completed to 100% of traffic.")
    return 1.00

live_sample = FROZEN_GOLDEN * 3   # stand-in for a larger live-traffic-shaped sample
final_stage = canary_rollout(PROMPTS_DIR / "system_prompt_v2.txt", live_sample)


## Exercises

**Exercise 1 (Warm-up):** Add a fifth item to `FROZEN_GOLDEN` designed to be hard for `system_prompt_v2` specifically (e.g. an arithmetic question where "concise, no work shown" hurts accuracy), and re-run `regression_gate`. Does the gate now fail?

**Exercise 2 (Apply):** Implement `diff_prompts(path_a, path_b) -> str` that returns a simple line-level diff between two prompt files (you can shell out to `difflib.unified_diff`), so a CI comment can show reviewers exactly what changed in plain text.

**Exercise 3 (Extend):** Notebook 27b built agent-trajectory evals with a solve-rate floor and a trajectory-score warning distinction. Sketch how `regression_gate` here would change for an AGENT prompt change instead of a Q&A prompt change — what should hard-block the merge versus just post a warning comment?


In [ ]:
# Exercise 1: Warm-up
# Task: Add a 5th golden item that's harder for the terser v2 prompt, re-run regression_gate.
# Hint: an arithmetic question with a tempting-but-wrong quick answer plays to v2's "concise" bias.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement diff_prompts(path_a, path_b) using difflib.unified_diff.
# Hint: difflib.unified_diff needs line lists, not raw strings — use .splitlines(keepends=True).

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch regression_gate's agent-prompt equivalent using notebook 27b's metrics.
# Hint: which of solve-rate / trajectory-score / cost-per-solved-task should BLOCK vs just WARN?

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
FROZEN_GOLDEN_V2 = FROZEN_GOLDEN + [
    {"q": "What is 17% of 240, rounded to the nearest whole number?", "ref": "41"},
]
# A "concise, don't show work" prompt is more likely to slip on a multi-step calculation like
# this one than the "double-check your answer" v1 prompt — re-running regression_gate with this
# item added may flip the verdict, illustrating why the frozen dataset's COVERAGE matters as
# much as its existence.

# Exercise 2
import difflib
def diff_prompts(path_a, path_b):
    a_lines = pathlib.Path(path_a).read_text().splitlines(keepends=True)
    b_lines = pathlib.Path(path_b).read_text().splitlines(keepends=True)
    return "".join(difflib.unified_diff(a_lines, b_lines, fromfile=str(path_a), tofile=str(path_b)))

print(diff_prompts(PROMPTS_DIR / "system_prompt_v1.txt", PROMPTS_DIR / "system_prompt_v2.txt"))

# Exercise 3
# For an agent prompt change:
#   - solve-rate regression -> HARD BLOCK (the agent objectively does its job worse; never merge)
#   - trajectory-score regression -> WARNING comment on the PR (efficiency dipped, but the task
#     still gets done — worth a reviewer's attention, not worth blocking on its own)
#   - cost-per-solved-task increase -> WARNING unless it crosses an explicit budget ceiling, in
#     which case it also hard-blocks (a correct-but-unaffordable agent is still a shipped failure)
```
</details>

## Key Takeaways
- A prompt is a production artifact — version it in a file, review it via a diff, never edit it invisibly in a database or a notebook cell.
- `regression_gate()` reuses notebook 24's exact eval-harness shape, just returning pass/fail instead of a printed score — that's the whole trick.
- The GitHub Actions job here is the same shape as this repo's own `.claude/checks/notebooks.sh` + `/loop` — do the work, run the check, block/repeat until it's green.
- A regression gate answers "is this safe to merge?"; a canary rollout answers "is this safe to give everyone, right now?" — stage the rollout and keep measuring at each stage.
- Distinguish hard-block metrics (correctness, safety) from warning-only metrics (efficiency, cost) so the gate blocks what actually matters without becoming so strict nobody can ship.

## What's Next
Tier 9 (notebook 34 onward) looks at frontier capabilities — computer use, voice/realtime — and the data-engineering skills that are this curriculum's unique edge.
